In [3]:
%load_ext autoreload
%autoreload 2

LANGAUGE = "ts"
FULL_REPO = "pancakeswap/pancake-frontend"
REPO = "pancake-frontend"
# Derived from original data set. 
AI_COMMITS_UNCLEAN = f"csv/{LANGAUGE}/{REPO}_AI_commits_unclean.csv"
# Derived from pydriller. Has THE AI as well
ALL_COMMITS_UNCLEAN = f"csv/{LANGAUGE}/{REPO}_all_commits_unclean.csv"

# Used to find start / end date of the dataset
MEREGED = f"csv/{LANGAUGE}/{REPO}_MEREGED.csv"

START_DATE = "2023-04-07 05:00:33+00:00"
END_DATE = "2024-01-24 06:22:16+00:00"


# ALL commits in time frame. 
WITHIN_TIMEFRAME = f"csv/{LANGAUGE}/{REPO}_within_time_frame.csv"
TIME_FRAME_AI_EXCLUDED = f"csv/{LANGAUGE}/{REPO}_AI_removed_in_timeframe.csv"
TIME_FRAME_AI_ONLY = f"csv/{LANGAUGE}/{REPO}_AI_in_timeframe.csv"





In [41]:
import pandas as pd
from pydriller import Repository
import os

unwanted_colums_in_original = ["branch", "committer", "author", "mention_language", "github_link", "mention"]

df = pd.read_csv("commit_mentions.csv")
df.drop(columns=unwanted_colums_in_original, inplace=True)
print(df.columns)


Index(['programming_language', 'repo_name', 'commit_id', 'tool'], dtype='str')


In [ ]:
# Amount of commits in original dataset. 
print(f"Length before cleaning the data: {len(df)}")


df = df[df["repo_name"].isin([FULL_REPO])]

print(f"Length after cleaning the data: {len(df)}")
if not os.path.isdir(f"csv/{LANGAUGE}/"):
    os.mkdir(f"csv/{LANGAUGE}")

# Saved from original dataset 
df.to_csv(AI_COMMITS_UNCLEAN, index=False)


Length before cleaning the data: 1572
Length after cleaning the data: 1003


In [43]:
data = []

for commit in Repository("https://github.com/pancakeswap/pancake-frontend").traverse_commits():
    data.append({
        "hash": commit.hash,
        "date": commit.author_date,
        "files": commit.files,
        "deletions": commit.deletions,
        "insertions": commit.insertions,
        "lines": commit.lines,

    })

df = pd.DataFrame(data)

# has EVERY commit in the repo
df.to_csv(ALL_COMMITS_UNCLEAN, index=False)

In [44]:
 # To get timeframe, update on top!
df1 = pd.read_csv(AI_COMMITS_UNCLEAN)   # contains commit_id
df2 = pd.read_csv(ALL_COMMITS_UNCLEAN)   # contains hash

merged = df1.merge(df2, left_on="commit_id", right_on="hash", how="inner")

merged.to_csv(MEREGED, index=False)
df = pd.read_csv(MEREGED)

df["date"] = pd.to_datetime(df["date"], utc=True)



df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)]


merged.to_csv(WITHIN_TIMEFRAME, index=False)


print("Min date:", df["date"].min())
print("Max date:", df["date"].max())

Min date: 2023-04-07 05:00:33+00:00
Max date: 2024-01-24 06:22:16+00:00


In [47]:
df1 = pd.read_csv(AI_COMMITS_UNCLEAN)   # contains commit_id
df2 = pd.read_csv(ALL_COMMITS_UNCLEAN)   # contains hash
df2 = df2[(df2["date"] >= START_DATE) & (df2["date"] <= END_DATE)]
merged = df2.merge(df1, left_on="hash", right_on="commit_id", how="left", indicator=True)

df2_unmatched = merged[merged["_merge"] == "left_only"]
df2_unmatched = df2_unmatched.drop(columns=["_merge", "programming_language","repo_name","commit_id","tool"])

print(f"Number of lines in NON AI, within timeframe! {len(df2_unmatched)}")
df2_unmatched.to_csv(TIME_FRAME_AI_EXCLUDED, index=False)



Number of lines in NON AI, within timeframe! 1115


In [48]:
df1 = pd.read_csv(AI_COMMITS_UNCLEAN)   # contains commit_id
df2 = pd.read_csv(ALL_COMMITS_UNCLEAN)  # contains hash

df2 = df2[(df2["date"] >= START_DATE) & (df2["date"] <= END_DATE)]

merged = df2.merge(df1, left_on="hash", right_on="commit_id", how="left", indicator=True)

df2_ai = merged[merged["_merge"] == "both"]
df2_ai = df2_ai.drop(columns=["_merge", "programming_language","repo_name","commit_id","tool"])

print(len(df2_ai))
df2_ai.to_csv(TIME_FRAME_AI_ONLY, index=False)

1001


In [8]:
from pydriller import Repository
from datetime import datetime, timedelta
import pandas as pd

# ---- CONFIG ----
REPO_URL = "https://github.com/pancakeswap/pancake-frontend"

START_DATE = datetime.fromisoformat("2023-04-07 05:00:33+00:00")
END_DATE   = datetime.fromisoformat("2024-01-24 06:22:16+00:00")

# ---- STORAGE ----
function_data = {}

# ---- MINING ----
for commit in Repository(
    REPO_URL,
    since=START_DATE,
    to=END_DATE
).traverse_commits():

    for mod in commit.modified_files:
        if not mod.new_path:
            continue

        if mod.methods:
            for m in mod.methods:
                key = (mod.new_path, m.name)

                # First time we see this function (within window)
                if key not in function_data:
                    function_data[key] = {
                        "file": mod.new_path,
                        "function": m.name,
                        "start_date": commit.author_date,
                        "changes": 0
                    }

                start = function_data[key]["start_date"]

                # Count changes within 30 days
                if commit.author_date <= start + timedelta(days=30):
                    function_data[key]["changes"] += 1

# ---- CONVERT TO PANDAS ----
rows = list(function_data.values())

df = pd.DataFrame(rows)

# Ensure datetime format
df["initial_commit_date"] = pd.to_datetime(df["start_date"], utc=True)

# Drop old column if you want cleaner output
df = df.drop(columns=["start_date"])

# Rename for clarity
df = df.rename(columns={
    "file": "file",
    "function": "function",
    "initial_commit_date": "initial_commit_date",
    "changes": "changes_in_30_days"
})

# ---- EXPORT ----
df.to_csv("function_changes.csv", index=False)

print("CSV file generated: function_changes.csv")

CSV file generated: function_changes.csv
